# Cookbook Experiment 5: Final Comparative Proof

## Comparison objective

This notebook consolidates evidence from all previous notebooks and answers the core experiment question:

- Does deep plain learning show weaker update flow?
- Is ResNet-style depth alone sufficient without skip addition?
- Does identity addition $F(x)+x$ improve layer-wise training dynamics?

## Comparison metrics used

1. Test accuracy and loss trajectory
2. Gradient ratio (first tracked conv / last tracked conv)
3. Final-epoch gradient and weight-delta snapshot at first and last tracked layers

## How to interpret

- A healthier model should avoid collapsing updates in early layers.
- Skip-enabled residual learning is expected to maintain stronger and more stable update signals across depth.

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt

paths = {
    'Simple CNN': 'outputs/01_simple_cnn/training_history.json',
    'ResNet No Skip': 'outputs/02_resnet_no_skip/training_history.json',
    'ResNet With Skip': 'outputs/03_resnet_with_skip/training_history.json',
}

hist = {}
for k, p in paths.items():
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing {p}. Run training notebooks first.')
    with open(p, 'r', encoding='utf-8') as f:
        hist[k] = json.load(f)
print('Loaded:', ', '.join(hist.keys()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, h in hist.items():
    axes[0].plot(h['test_acc'], linewidth=2, label=name)
    axes[1].plot(h['test_loss'], linewidth=2, label=name)
axes[0].set_title('Test Accuracy')
axes[1].set_title('Test Loss')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel('Acc (%)')
axes[1].set_ylabel('Loss')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
for name, h in hist.items():
    first = h['tracked_layers'][0]
    last = h['tracked_layers'][-1]
    f = np.array(h['epoch_grad_norms'][first], dtype=float)
    l = np.array(h['epoch_grad_norms'][last], dtype=float)
    ratio = f / np.maximum(l, 1e-12)
    plt.plot(ratio, marker='o', label=name)
plt.title('Gradient Ratio (first conv / last conv)')
plt.xlabel('Epoch')
plt.ylabel('Ratio')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print('Final comparison snapshot')
for name, h in hist.items():
    first = h['tracked_layers'][0]
    last = h['tracked_layers'][-1]
    fg = h['epoch_grad_norms'][first][-1]
    lg = h['epoch_grad_norms'][last][-1]
    fd = h['weight_deltas'][first][-1]
    ld = h['weight_deltas'][last][-1]
    best = max(h['test_acc'])
    print(f'\n{name}')
    print(f'  best test acc: {best:.3f}')
    print(f'  grad first/last: {fg:.6f} / {lg:.6f}')
    print(f'  delta first/last: {fd:.6f} / {ld:.6f}')